# Transfer Volume (Dashboard Migration)

Copies the source export volume contents to the target import volume (same metastore). Run on the **target** workspace after the source has written inventory, exported, and transformed artifacts.

## Configuration

Read widget parameters: source/target catalogs, schema, and volume names. Derive the export and import volume paths.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog (holds export volume)")
dbutils.widgets.text("target_catalog", "", "Target catalog (holds import volume)")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("export_volume", "dashboard_migration", "Export volume name")
dbutils.widgets.text("import_volume", "dashboard_migration", "Import volume name")

SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
EXPORT_VOLUME = dbutils.widgets.get("export_volume").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()

if not SOURCE_CATALOG or not TARGET_CATALOG:
    raise ValueError("Set source_catalog and target_catalog")

SRC_PATH = f"/Volumes/{SOURCE_CATALOG}/{SCHEMA}/{EXPORT_VOLUME}"
TGT_PATH = f"/Volumes/{TARGET_CATALOG}/{SCHEMA}/{IMPORT_VOLUME}"
print(f"Source: {SRC_PATH}")
print(f"Target: {TGT_PATH}")

## Verify export data exists

Checks that the export volume exists and contains at least one expected subdirectory (transformed or exported) before proceeding.

In [ ]:
import os
expected_subdirs = ["transformed", "exported", "dashboard_inventory_approved", "dashboard_inventory"]
has_data = os.path.isdir(SRC_PATH) and any(
    os.path.isdir(os.path.join(SRC_PATH, d)) for d in expected_subdirs
)
if not has_data:
    print("No export data at source; skipping transfer.")
    dbutils.notebook.exit("SKIP_NO_SOURCE")

## Create import volume and copy

Creates the import volume if needed, clears the target path, then copies all export files and subdirs to the target volume.

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TARGET_CATALOG}.{SCHEMA}.{IMPORT_VOLUME}")

import shutil

# Use DBFS path for volume when using os.makedirs / listdir
SRC_DFS = SRC_PATH.replace("/Volumes", "/dbfs/Volumes")
TGT_DFS = TGT_PATH.replace("/Volumes", "/dbfs/Volumes")
os.makedirs(TGT_DFS, exist_ok=True)

if os.path.exists(TGT_DFS):
    for item in os.listdir(TGT_DFS):
        p = os.path.join(TGT_DFS, item)
        if os.path.isdir(p):
            shutil.rmtree(p)
        else:
            os.remove(p)

count_dirs = count_files = 0
for entry in sorted(os.listdir(SRC_DFS)):
    src = os.path.join(SRC_DFS, entry)
    dst = os.path.join(TGT_DFS, entry)
    if os.path.isdir(src):
        shutil.copytree(src, dst)
        count_dirs += 1
        print(f"  {entry}/ copied")
    else:
        shutil.copy2(src, dst)
        count_files += 1
        print(f"  {entry} copied")

print(f"TRANSFERRED {count_dirs} dir(s), {count_files} file(s)")